#### Comparative Analysis of Emotional Outcomes

In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import os

def plot_emotion_comparison(emotion_results_path, labels_path, target_label, save_root_path, task_type='Coping'):
    """
    Compare the emotion distribution of speech segments across different clinical labels.
    
    Args:
        emotion_results_path (str): Path to the JSON file containing emotion detection results.
        labels_path (str): Path to the JSON file containing subjects' ground truth labels.
        target_label (str): The binary classification label to compare, e.g., 'is_depression'.
        save_root_path (str): Root directory path to save the generated charts.
        task_type (str): The task type, e.g., 'Coping' or 'Training'.
    """
    print(f"Analyzing emotion distribution for '{target_label}' in the '{task_type}' task...")
    
    # 1. Load data
    if not os.path.exists(emotion_results_path) or not os.path.exists(labels_path):
        print(f"Error: Specified JSON files not found. Please check the paths:\n  - {emotion_results_path}\n  - {labels_path}")
        return
        
    with open(emotion_results_path, 'r', encoding='utf-8') as f:
        emotion_data = json.load(f)

    with open(labels_path, 'r', encoding='utf-8') as f:
        labels_data = json.load(f)

    # 2. Extract subjects' ground truth labels
    # The labels data uses a cross-validation format. We iterate through 'outer_test' 
    # to collect labels for all subjects.
    patient_labels = {}
    if target_label not in labels_data:
        print(f"Error: Target label '{target_label}' not found in the labels file.")
        return
        
    for fold in labels_data[target_label]:
        outer_test = fold.get('outer_test', {})
        for label_val in ['0', '1']:
            if label_val in outer_test:
                for patient_id in outer_test[label_val].keys():
                    patient_labels[patient_id] = int(label_val)

    # 3. Calculate segment emotion distribution for different groups
    # Dictionary structure: { 0: {'neu': 100, 'sad': 50...}, 1: {'neu': 80, 'sad': 120...} }
    emotion_counts = {0: defaultdict(int), 1: defaultdict(int)}
    
    for patient_id, tasks in emotion_data.items():
        if patient_id not in patient_labels:
            continue # Skip if the subject doesn't have the current label
            
        p_label = patient_labels[patient_id]
        
        if task_type in tasks:
            # Parse the structure of All_Emotion_Results: [{"1.wav": {"emotion": "sad"...}}, ...]
            for segment_info in tasks[task_type]:
                for wav_name, result in segment_info.items():
                    pred_emotion = result.get('emotion')
                    if pred_emotion:
                        emotion_counts[p_label][pred_emotion] += 1

    # 4. Calculate percentages
    emotions = ['neu', 'sad', 'hap', 'ang'] # Typical output classes for SpeechBrain
    display_names = ['Neutral', 'Sadness', 'Happiness', 'Anger']
    
    group_0_pct = []
    group_1_pct = []

    total_0 = sum(emotion_counts[0].values())
    total_1 = sum(emotion_counts[1].values())
    
    if total_0 == 0 or total_1 == 0:
        print(f"Warning: Counted data is empty for {target_label} in {task_type}. Check if labels or task names match.")
        return

    for emo in emotions:
        pct_0 = (emotion_counts[0][emo] / total_0 * 100) if total_0 > 0 else 0
        pct_1 = (emotion_counts[1][emo] / total_1 * 100) if total_1 > 0 else 0
        group_0_pct.append(pct_0)
        group_1_pct.append(pct_1)

    # 5. Visualization - Plot grouped bar chart
    x = np.arange(len(emotions))
    width = 0.35  # Bar width

    fig, ax = plt.subplots(figsize=(8, 6), dpi=120)
    rects1 = ax.bar(x - width/2, group_0_pct, width, label=f'Negative (0) [n={total_0} segments]', color='#4C72B0')
    rects2 = ax.bar(x + width/2, group_1_pct, width, label=f'Positive (1) [n={total_1} segments]', color='#C44E52')

    # Enhance chart aesthetics
    label_title = target_label.replace('is_', '').capitalize()
    ax.set_ylabel('Percentage of Audio Segments (%)', fontsize=12)
    ax.set_title(f'Segment Mood Distribution by {label_title} ({task_type} Task)', fontsize=14, pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(display_names, fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Add specific value labels on top of the bars
    ax.bar_label(rects1, fmt='%.1f%%', padding=3, fontsize=9)
    ax.bar_label(rects2, fmt='%.1f%%', padding=3, fontsize=9)

    fig.tight_layout()
    
    # Save automatically based on label name
    save_name = os.path.join(save_root_path, f"{task_type}_{target_label}_mood_distribution.png")
    plt.savefig(save_name, dpi=300, bbox_inches='tight') 
    
    print(f"Chart successfully saved as: {save_name}\n")
    plt.close() # Close the image in memory after saving to prevent overlapping

# ==========================================
# Execution Block
# ==========================================
if __name__ == "__main__":
    
    # ==========================================
    # Unified Parameters Configuration Entry
    # ==========================================
    CONFIG = {
        "EMOTION_RESULTS_PATH": "./data/All_Emotion_Results.json",
        "SAVE_ROOT_PATH": "./Emotion_Analysis_Results",
        # Use {task_type} as a placeholder to dynamically load labels for different tasks
        "LABELS_FILE_TEMPLATE": "./data/datasets/{task_type}/{task_type}_Split.json",
        
        "TASKS": ['Coping', 'Training'],
        "TARGET_LABELS": ['is_depression', 'is_agitation', 'is_retardation', 'is_HRSD']
    }
    
    # Ensure the root directory for saving charts exists
    os.makedirs(CONFIG["SAVE_ROOT_PATH"], exist_ok=True)
    
    # Run the analysis loop
    for task_type in CONFIG["TASKS"]:
        # Format the specific label file path for the current task
        labels_path = CONFIG["LABELS_FILE_TEMPLATE"].format(task_type=task_type)
        
        for target_label in CONFIG["TARGET_LABELS"]:
            plot_emotion_comparison(
                emotion_results_path=CONFIG["EMOTION_RESULTS_PATH"], 
                labels_path=labels_path, 
                target_label=target_label,
                save_root_path=CONFIG["SAVE_ROOT_PATH"],
                task_type=task_type
            )

Analyzing emotion distribution for 'is_depression' in the 'Coping' task...
Chart successfully saved as: ./Emotion_Analysis_Results\Coping_is_depression_mood_distribution.png

Analyzing emotion distribution for 'is_agitation' in the 'Coping' task...
Chart successfully saved as: ./Emotion_Analysis_Results\Coping_is_agitation_mood_distribution.png

Analyzing emotion distribution for 'is_retardation' in the 'Coping' task...
Chart successfully saved as: ./Emotion_Analysis_Results\Coping_is_retardation_mood_distribution.png

Analyzing emotion distribution for 'is_HRSD' in the 'Coping' task...
Chart successfully saved as: ./Emotion_Analysis_Results\Coping_is_HRSD_mood_distribution.png

Analyzing emotion distribution for 'is_depression' in the 'Training' task...
Chart successfully saved as: ./Emotion_Analysis_Results\Training_is_depression_mood_distribution.png

Analyzing emotion distribution for 'is_agitation' in the 'Training' task...
Chart successfully saved as: ./Emotion_Analysis_Results\T